### Reinforcement Learning for Petrophysicists

#### Keith R. Holdaway FGS

In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime

class ReservoirEnvironment:
    """Simple reservoir environment for RL training"""
    
    def __init__(self, grid_size=5):
        self.grid_size = grid_size
        # Simulate porosity values (0-1) across the reservoir
        self.porosity_grid = np.random.uniform(0.1, 0.3, (grid_size, grid_size))
        # Add some high-porosity sweet spots
        self.porosity_grid[2, 2] = 0.8  # Sweet spot 1
        self.porosity_grid[0, 4] = 0.7  # Sweet spot 2
        self.porosity_grid[4, 1] = 0.75 # Sweet spot 3
        
        self.current_position = [0, 0]  # Start at top-left
        self.visited_positions = set()
        
    def reset(self):
        """Reset environment to initial state"""
        self.current_position = [0, 0]
        self.visited_positions = set()
        return tuple(self.current_position)
    
    def step(self, action):
        """Take an action and return new state, reward, done"""
        x, y = self.current_position
        
        # Action mapping: 0=up, 1=right, 2=down, 3=left
        if action == 0 and y > 0: y -= 1
        elif action == 1 and x < self.grid_size-1: x += 1
        elif action == 2 and y < self.grid_size-1: y += 1
        elif action == 3 and x > 0: x -= 1
        
        self.current_position = [x, y]
        state = tuple(self.current_position)
        
        # Calculate reward based on porosity
        porosity = self.porosity_grid[y, x]
        reward = porosity * 10  # Scale reward
        
        # Penalize for revisiting positions to encourage exploration
        if state in self.visited_positions:
            reward -= 2
        else:
            self.visited_positions.add(state)
        
        # Episode ends when we find high porosity (>0.6) or after many steps
        done = (porosity > 0.6) or (len(self.visited_positions) >= 15)
        
        return state, reward, done

class QLearningAgent:
    """Q-learning agent for reservoir navigation"""
    
    def __init__(self, grid_size=5, learning_rate=0.1, discount_factor=0.9, 
                 exploration_rate=1.0, exploration_decay=0.995):
        self.grid_size = grid_size
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = exploration_rate
        self.epsilon_decay = exploration_decay
        self.epsilon_min = 0.01
        
        # Initialize Q-table with zeros
        self.q_table = np.zeros((grid_size, grid_size, 4))  # states × actions
        
    def choose_action(self, state):
        """Choose action using epsilon-greedy policy"""
        x, y = state
        
        # Exploration: random action
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        
        # Exploitation: best known action
        return np.argmax(self.q_table[y, x, :])
    
    def update_q_value(self, state, action, reward, next_state):
        """Update Q-value using Q-learning formula"""
        x, y = state
        next_x, next_y = next_state
        
        current_q = self.q_table[y, x, action]
        max_future_q = np.max(self.q_table[next_y, next_x, :])
        
        # Q-learning formula
        new_q = current_q + self.lr * (reward + self.gamma * max_future_q - current_q)
        self.q_table[y, x, action] = new_q
        
        # Decay exploration rate
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

def train_agent(episodes=1000):
    """Train the Q-learning agent"""
    env = ReservoirEnvironment()
    agent = QLearningAgent()
    
    # Store training data
    training_data = []
    
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        steps = 0
        
        while True:
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.update_q_value(state, action, reward, next_state)
            
            total_reward += reward
            steps += 1
            state = next_state
            
            if done:
                break
        
        # Record episode results
        training_data.append({
            'episode': episode,
            'total_reward': total_reward,
            'steps': steps,
            'exploration_rate': agent.epsilon,
            'average_reward_per_step': total_reward / steps if steps > 0 else 0
        })
        
        if episode % 100 == 0:
            print(f"Episode {episode}: Reward = {total_reward:.2f}, Steps = {steps}, Epsilon = {agent.epsilon:.3f}")
    
    return env, agent, training_data

def save_results(env, agent, training_data):
    """Save all results to CSV files with detailed headers and explanations"""
    
    # Create metadata header for all files
    metadata = {
        'simulation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'grid_size': env.grid_size,
        'learning_rate': agent.lr,
        'discount_factor': agent.gamma,
        'initial_exploration_rate': 1.0,
        'final_exploration_rate': agent.epsilon,
        'total_episodes': len(training_data)
    }
    
    # 1. Save porosity grid with detailed headers
    porosity_df = pd.DataFrame(env.porosity_grid)
    porosity_df.columns = [f'X_{i}' for i in range(env.grid_size)]
    porosity_df.index = [f'Y_{i}' for i in range(env.grid_size)]
    
    # Add metadata as comments
    porosity_comments = [
        f"# RESERVOIR POROSITY GRID - Reinforcement Learning Simulation",
        f"# Generated: {metadata['simulation_date']}",
        f"# Grid Size: {metadata['grid_size']}x{metadata['grid_size']}",
        f"# Description: Simulated porosity values (0-1) representing reservoir characteristics",
        f"# Interpretation: Higher values indicate better reservoir quality (sweet spots)",
        f"# X-axis: East-West direction, Y-axis: North-South direction",
        f"# Values range: 0.1 (poor) to 0.8 (excellent porosity)",
        f"# Sweet spots: Locations with porosity > 0.6 represent optimal drilling targets"
    ]
    
    with open('porosity_grid.csv', 'w') as f:
        for comment in porosity_comments:
            f.write(comment + '\n')
        porosity_df.to_csv(f, index=True)
    print("Saved porosity grid to porosity_grid.csv")
    
    # 2. Save Q-table with detailed explanations
    q_table_flat = []
    for y in range(agent.grid_size):
        for x in range(agent.grid_size):
            for action in range(4):
                action_name = ['Move_North', 'Move_East', 'Move_South', 'Move_West'][action]
                best_action = np.argmax(agent.q_table[y, x, :]) == action
                q_table_flat.append({
                    'x_coordinate': x,
                    'y_coordinate': y,
                    'action': action_name,
                    'q_value': agent.q_table[y, x, action],
                    'is_optimal_action': best_action,
                    'porosity_at_location': env.porosity_grid[y, x]
                })
    
    q_df = pd.DataFrame(q_table_flat)
    q_comments = [
        f"# Q-TABLE - Learned Value Function",
        f"# Generated: {metadata['simulation_date']}",
        f"# Description: Q-values represent expected long-term rewards for each action at each location",
        f"# Interpretation: Higher Q-values indicate better actions. The AI learns which moves lead to better reservoir zones",
        f"# Columns:",
        f"#   - x_coordinate, y_coordinate: Position in the reservoir grid",
        f"#   - action: Possible movement direction",
        f"#   - q_value: Expected cumulative reward (higher = better)",
        f"#   - is_optimal_action: TRUE if this is the best action for this location",
        f"#   - porosity_at_location: Porosity value at this position for reference"
    ]
    
    with open('q_table.csv', 'w') as f:
        for comment in q_comments:
            f.write(comment + '\n')
        q_df.to_csv(f, index=False)
    print("Saved Q-table to q_table.csv")
    
    # 3. Save training history with performance metrics
    training_df = pd.DataFrame(training_data)
    # Add moving averages for better trend analysis
    training_df['reward_ma_10'] = training_df['total_reward'].rolling(window=10).mean()
    training_df['reward_ma_50'] = training_df['total_reward'].rolling(window=50).mean()
    
    training_comments = [
        f"# TRAINING HISTORY - Learning Progress Over Time",
        f"# Generated: {metadata['simulation_date']}",
        f"# Description: Records the AI's learning progress through each training episode",
        f"# Interpretation: Shows how the agent improves at finding high-porosity zones over time",
        f"# Learning Parameters: LR={metadata['learning_rate']}, DF={metadata['discount_factor']}",
        f"# Columns:",
        f"#   - episode: Training episode number",
        f"#   - total_reward: Cumulative reward earned in the episode",
        f"#   - steps: Number of moves taken",
        f"#   - exploration_rate: Probability of random exploration (decays over time)",
        f"#   - average_reward_per_step: Efficiency metric (higher = better)",
        f"#   - reward_ma_10: 10-episode moving average of rewards",
        f"#   - reward_ma_50: 50-episode moving average of rewards"
    ]
    
    with open('training_history.csv', 'w') as f:
        for comment in training_comments:
            f.write(comment + '\n')
        training_df.to_csv(f, index=False)
    print("Saved training history to training_history.csv")
    
    # 4. Save optimal policy
    policy_grid = np.empty((agent.grid_size, agent.grid_size), dtype=object)
    policy_values = np.zeros((agent.grid_size, agent.grid_size))
    
    for y in range(agent.grid_size):
        for x in range(agent.grid_size):
            best_action = np.argmax(agent.q_table[y, x, :])
            policy_grid[y, x] = ['↑ North', '→ East', '↓ South', '← West'][best_action]
            policy_values[y, x] = np.max(agent.q_table[y, x, :])
    
    policy_df = pd.DataFrame(policy_grid)
    policy_df.columns = [f'X_{i}' for i in range(env.grid_size)]
    policy_df.index = [f'Y_{i}' for i in range(env.grid_size)]
    
    policy_comments = [
        f"# OPTIMAL POLICY - Learned Navigation Strategy",
        f"# Generated: {metadata['simulation_date']}",
        f"# Description: Shows the best movement direction from each location in the reservoir",
        f"# Interpretation: The AI's recommended path to find high-porosity zones efficiently",
        f"# Usage: Start at any position and follow the arrows to navigate to sweet spots",
        f"# Arrow Directions:",
        f"#   - ↑ North: Move upward (decreasing Y coordinate)",
        f"#   - → East: Move right (increasing X coordinate)", 
        f"#   - ↓ South: Move downward (increasing Y coordinate)",
        f"#   - ← West: Move left (decreasing X coordinate)"
    ]
    
    with open('optimal_policy.csv', 'w') as f:
        for comment in policy_comments:
            f.write(comment + '\n')
        policy_df.to_csv(f, index=True)
    print("Saved optimal policy to optimal_policy.csv")
    
    # 5. Save summary report
    save_summary_report(env, agent, training_data, metadata)

def save_summary_report(env, agent, training_data, metadata):
    """Create a comprehensive summary report"""
    
    final_rewards = [episode['total_reward'] for episode in training_data[-100:]]  # Last 100 episodes
    avg_final_reward = np.mean(final_rewards)
    
    summary_data = {
        'metric': [
            'Simulation Date',
            'Grid Size',
            'Total Training Episodes',
            'Learning Rate',
            'Discount Factor',
            'Final Exploration Rate',
            'Average Final Reward (last 100 episodes)',
            'Maximum Porosity in Reservoir',
            'Sweet Spots Found (>0.6 porosity)',
            'Training Efficiency Improvement',
            'Best Learned Path Length'
        ],
        'value': [
            metadata['simulation_date'],
            f"{metadata['grid_size']}x{metadata['grid_size']}",
            metadata['total_episodes'],
            metadata['learning_rate'],
            metadata['discount_factor'],
            f"{agent.epsilon:.4f}",
            f"{avg_final_reward:.2f}",
            f"{np.max(env.porosity_grid):.3f}",
            f"{np.sum(env.porosity_grid > 0.6)} locations",
            f"{(avg_final_reward - training_data[0]['total_reward'])/training_data[0]['total_reward']*100:.1f}%",
            f"~{np.mean([episode['steps'] for episode in training_data[-20:]]):.1f} steps"
        ],
        'interpretation': [
            'Date and time of simulation run',
            'Size of reservoir grid being explored',
            'Total number of training iterations',
            'How quickly the AI adapts to new information',
            'Importance of future rewards vs immediate rewards',
            'Final probability of random exploration',
            'Average performance after training completion',
            'Best reservoir quality found in simulation',
            'Number of high-quality drilling targets',
            'Percentage improvement from start to end of training',
            'Average steps needed to find sweet spots'
        ]
    }
    
    summary_df = pd.DataFrame(summary_data)
    
    summary_comments = [
        f"# REINFORCEMENT LEARNING SUMMARY REPORT - Petrophysics Application",
        f"# SIMULATION OVERVIEW",
        f"# This report summarizes the AI's learning performance in reservoir navigation",
        f"# The agent learned to find optimal paths to high-porosity zones using Q-learning",
        f"# KEY INSIGHTS:",
        f"# - The AI started with random exploration and gradually developed an optimal strategy",
        f"# - Higher discount factors prioritize long-term rewards over immediate gains",
        f"# - The exploration rate decayed as the agent became more confident in its decisions",
        f"# PETROPHYSICS APPLICATION:",
        f"# This demonstrates how RL can optimize well placement and reservoir navigation",
        f"# Similar principles apply to real-world drilling optimization and reservoir characterization"
    ]
    
    with open('learning_summary.csv', 'w') as f:
        for comment in summary_comments:
            f.write(comment + '\n')
        summary_df.to_csv(f, index=False)
    print("Saved summary report to learning_summary.csv")

def demonstrate_optimal_path(env, agent):
    """Show the optimal path learned by the agent"""
    state = env.reset()
    path = [state]
    total_reward = 0
    
    print("\n" + "="*60)
    print("OPTIMAL PATH DEMONSTRATION")
    print("="*60)
    print(f"Starting at position (X={state[0]}, Y={state[1]})")
    print("Following the learned optimal policy...\n")
    
    for step in range(20):  # Prevent infinite loops
        x, y = state
        action = np.argmax(agent.q_table[y, x, :])
        action_name = ['North', 'East', 'South', 'West'][action]
        
        state, reward, done = env.step(action)
        path.append(state)
        total_reward += reward
        
        porosity = env.porosity_grid[state[1], state[0]]
        porosity_quality = "POOR" if porosity < 0.2 else "FAIR" if porosity < 0.4 else "GOOD" if porosity < 0.6 else "EXCELLENT"
        
        print(f"Step {step+1:2d}: Move {action_name:5s} to (X={state[0]}, Y={state[1]}) | "
              f"Porosity: {porosity:.3f} ({porosity_quality:8s}) | Reward: {reward:6.2f}")
        
        if done:
            if porosity > 0.6:
                print(f"\n🎯 SWEET SPOT FOUND! High-porosity zone reached.")
            else:
                print(f"\n⏱️  Maximum steps reached.")
            print(f"📊 Total reward: {total_reward:.2f}")
            break
    
    return path, total_reward

# Main execution
if __name__ == "__main__":
    print("REINFORCEMENT LEARNING FOR PETROPHYSICS")
    print("="*50)
    print("Training Q-learning agent for reservoir navigation...")
    print("This simulates finding optimal paths to high-porosity zones.\n")
    
    # Train the agent
    env, agent, training_data = train_agent(episodes=500)
    
    # Save results to CSV files with detailed explanations
    save_results(env, agent, training_data)
    
    # Demonstrate learned behavior
    path, final_reward = demonstrate_optimal_path(env, agent)
    
    print(f"\n" + "="*50)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("="*50)
    print(f"📈 Final exploration rate: {agent.epsilon:.3f}")
    print(f"💰 Optimal path reward: {final_reward:.2f}")
    print(f"🕒 Average steps to target: {np.mean([episode['steps'] for episode in training_data[-20:]]):.1f}")
    
    print("\n📁 GENERATED OUTPUT FILES:")
    print("   → porosity_grid.csv    - Reservoir porosity distribution")
    print("   → q_table.csv          - Learned action values (Q-values)") 
    print("   → training_history.csv - Learning progress over time")
    print("   → optimal_policy.csv   - Best movement strategy")
    print("   → learning_summary.csv - Comprehensive performance report")
    
    print("\n💡 PETROPHYSICS INTERPRETATION:")
    print("   Each file contains detailed headers explaining how to interpret")
    print("   the results in the context of reservoir characterization and")
    print("   well placement optimization.")

REINFORCEMENT LEARNING FOR PETROPHYSICS
Training Q-learning agent for reservoir navigation...
This simulates finding optimal paths to high-porosity zones.

Episode 0: Reward = 24.81, Steps = 41, Epsilon = 0.814
Episode 100: Reward = 14.97, Steps = 4, Epsilon = 0.017
Episode 200: Reward = 14.97, Steps = 4, Epsilon = 0.010
Episode 300: Reward = 14.97, Steps = 4, Epsilon = 0.010
Episode 400: Reward = 14.97, Steps = 4, Epsilon = 0.010
Saved porosity grid to porosity_grid.csv
Saved Q-table to q_table.csv
Saved training history to training_history.csv
Saved optimal policy to optimal_policy.csv
Saved summary report to learning_summary.csv

OPTIMAL PATH DEMONSTRATION
Starting at position (X=0, Y=0)
Following the learned optimal policy...

Step  1: Move South to (X=0, Y=1) | Porosity: 0.181 (POOR    ) | Reward:   1.81
Step  2: Move East  to (X=1, Y=1) | Porosity: 0.245 (FAIR    ) | Reward:   2.45
Step  3: Move South to (X=1, Y=2) | Porosity: 0.271 (FAIR    ) | Reward:   2.71
Step  4: Move East 

### Generated CSV Files Explained
#### 1. porosity_grid.csv
Contains simulated porosity values (0.1-0.8) across the reservoir

Represents the "environment" your agent explores

High values (>0.6) represent sweet spots

#### 2. q_table.csv
The core of what the agent learns

Contains Q-values for each (position, action) combination

Higher values indicate better long-term decisions

#### 3. training_history.csv
Tracks learning progress over episodes

Shows how total rewards improve as the agent learns

Documents exploration rate decay over time

#### 4. optimal_policy.csv
Shows the best action (↑→↓←) for each position

#### 5. learning_summary.csv
Shows comprehensive performance report